<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/main/corrected_one_step_robustness_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Corrected one-step model: split verification, uncertainty, and reward sensitivity

**Run after cells 0–10** of `corrected_one_step_qlearning_sarsa.ipynb` (the original notebook's validation evaluation). Do not run its final-test cell 12 until selecting and freezing settings using validation. This supplement uses the original notebook's in-memory variables, never tunes on the test set.

**Interpretation:** all results are estimated, not causal operational gains. With terminal one-step episodes and `GAMMA=0`, Q-learning and SARSA have identical mathematical update targets. Algorithm differences in this experiment arise from sampling/exploration and finite training, not sequential bootstrapping.

## A. Verify the evaluation split and candidate coverage

In [1]:
assert {'train','validation','test'} <= set(frames)
assert len(val)==len(frames['validation'])
print('Validation rows:',len(val),'Test rows:',len(test))
print('Previously reported table (validation):')
display(validation_summary)
print('Validation summary was created by evaluate_policy(val, val_candidates, ...).')
print('Final test is evaluated separately by cell 12 of the original corrected notebook.')

NameError: name 'frames' is not defined

## B. Helpers: independent seeds, one-step training, and paired evaluation

In [ ]:
import numpy as np, pandas as pd

# Use existing train_candidates, val_candidates, STATES, STATE_TO_ID, MODE_TO_ID,
# n_states, n_actions, ALPHA, EPISODES, EPS_START, EPS_END, MODE, and train.
# All outcomes below are already estimated by the TRAIN-fitted estimator.

def add_reward(df, days_weight, late_weight):
    out=df.copy()
    out['reward']=out['profit']-days_weight*out['days']-late_weight*out['late']
    return out

def train_repeated(candidates, algorithm, seed, epochs=EPISODES):
    assert algorithm in ('Q-learning','SARSA')
    rng=np.random.default_rng(seed)
    Q=np.zeros((n_states,n_actions),dtype=float)
    z=candidates.copy()
    z['state_id']=z['state'].map(STATE_TO_ID)
    z=z[z['state_id'].notna()].copy()
    z['state_id']=z['state_id'].astype(int)
    by_order={int(i):g[['state_id','action_id','reward']].to_numpy(float)
              for i,g in z.groupby('row_id',sort=False)}
    ids=np.array(sorted(by_order),dtype=int)
    for epoch in range(epochs):
        eps=EPS_END+(EPS_START-EPS_END)*(1-epoch/max(epochs-1,1))
        for order_id in rng.permutation(ids):
            choices=by_order[int(order_id)]
            s=int(choices[0,0]); actions=choices[:,1].astype(int)
            if rng.random()<eps:
                a=int(rng.choice(actions))
            else:
                a=int(actions[np.argmax(Q[s,actions])])
            r=float(choices[np.flatnonzero(actions==a)[0],2])
            # Both updates reduce to immediate reward because every step is terminal.
            Q[s,a]+=ALPHA*(r-Q[s,a])
    return Q

def paired_for(Q,candidates,orders):
    # Identical eligibility and matched historical-action benchmark for every policy.
    z=candidates.copy()
    z['state_id']=z['state'].map(STATE_TO_ID)
    z=z[z.state_id.notna()].copy()
    z['state_id']=z['state_id'].astype(int)
    z['q']=Q[z.state_id.to_numpy(),z.action_id.to_numpy()]
    chosen=(z.sort_values(['row_id','q','action_id'],ascending=[True,False,True])
             .drop_duplicates('row_id'))
    recorded=orders.iloc[z.row_id.to_numpy()][MODE].to_numpy()
    observed=z[z['mode'].to_numpy()==recorded].drop_duplicates('row_id')
    return chosen[['row_id','mode','profit','days','late','reward']].merge(
        observed[['row_id','profit','days','late','reward']],on='row_id',
        how='inner',suffixes=('_policy','_historical')).sort_values('row_id').reset_index(drop=True)

def summarize_paired(p,algorithm,seed,wd,wl):
    d={'algorithm':algorithm,'seed':seed,'days_weight':wd,'late_weight':wl,
       'eligible_orders':len(p),'coverage':len(p)/len(val)}
    for m in ['profit','days','late','reward']:
        d['historical_'+m]=p[m+'_historical'].mean()
        d['policy_'+m]=p[m+'_policy'].mean()
        d['delta_'+m]=d['policy_'+m]-d['historical_'+m]
    return d

## C. Paired bootstrap: Q-learning versus SARSA on the same orders

This measures sampling uncertainty **conditional on the two trained policies and the historical outcome model**. It does not measure estimator uncertainty, causal effects, or training-seed variation.

In [ ]:
def bootstrap_difference(q_pairs,s_pairs,metric,draws=2000,seed=2026):
    q=q_pairs[['row_id',metric+'_policy']].rename(columns={metric+'_policy':'q'})
    s=s_pairs[['row_id',metric+'_policy']].rename(columns={metric+'_policy':'s'})
    joined=q.merge(s,on='row_id',how='inner',validate='one_to_one')
    diff=(joined['q']-joined['s']).to_numpy(float)
    if len(diff)<2: raise ValueError('Insufficient paired orders')
    rng=np.random.default_rng(seed)
    # Memory-efficient bootstrap (do not allocate draws x number_of_orders).
    means=np.empty(draws)
    for j in range(draws):
        means[j]=diff[rng.integers(0,len(diff),len(diff))].mean()
    lo,hi=np.percentile(means,[2.5,97.5])
    return {'metric':metric,'paired_orders':len(diff),'q_minus_s':diff.mean(),
            'ci_95_low':lo,'ci_95_high':hi,
            'ci_excludes_zero':bool(lo>0 or hi<0)}

paired_ci=pd.DataFrame([bootstrap_difference(val_ql,val_sarsa,m) for m in ['profit','days','late','reward']])
display(paired_ci)
print('For profit/reward, positive Q−SARSA favors Q. For days/late, negative favors Q.')

## D. Reward-weight sensitivity and multiple independent seeds

Weights are scenario choices, not fitted causal coefficients. Compare all candidates on validation. Choose a configuration **before** opening the test results. Five seeds provide an initial robustness check, not a definitive statistical analysis.

In [ ]:
# Adjust these only based on validation and record any change in the dissertation.
WEIGHT_SCENARIOS=[
    ('original',0.5,2.0),
    ('moderate_late',0.5,20.0),
    ('strong_late',0.5,50.0),
    ('very_strong_late',0.5,100.0),
]
SEEDS=[42,101,202,303,404]
records=[]
for scenario,wd,wl in WEIGHT_SCENARIOS:
    tc=add_reward(train_candidates,wd,wl)
    vc=add_reward(val_candidates,wd,wl)
    for seed in SEEDS:
        for alg,offset in [('Q-learning',0),('SARSA',100000)]:
            Q=train_repeated(tc,alg,seed+offset)
            paired=paired_for(Q,vc,val)
            rec=summarize_paired(paired,alg,seed,wd,wl)
            rec['scenario']=scenario
            records.append(rec)
            print(scenario,alg,'seed',seed,'n=',len(paired),
                  'Δprofit=',round(rec['delta_profit'],3),
                  'Δdays=',round(rec['delta_days'],3),
                  'Δlate=',round(rec['delta_late'],4))
seed_results=pd.DataFrame(records)
seed_results.to_csv('corrected_validation_seed_weight_results.csv',index=False)
summary=(seed_results.groupby(['scenario','algorithm','days_weight','late_weight'])
         .agg(n_seeds=('seed','nunique'),orders=('eligible_orders','min'),
              mean_delta_profit=('delta_profit','mean'),sd_delta_profit=('delta_profit','std'),
              mean_delta_days=('delta_days','mean'),sd_delta_days=('delta_days','std'),
              mean_delta_late=('delta_late','mean'),sd_delta_late=('delta_late','std'),
              mean_delta_reward=('delta_reward','mean')).reset_index())
summary['all_three_improve']=(summary.mean_delta_profit>0)&(summary.mean_delta_days<0)&(summary.mean_delta_late<0)
display(summary)
summary.to_csv('corrected_validation_weight_sensitivity.csv',index=False)

In [ ]:
# Final selected reward configuration
# Based on validation results only

days_weight = 0.5
late_weight = 50.0
SEED = 42

print("Selected late penalty:", late_weight)
print("Selected delivery penalty:", days_weight)
print("Selected training seed:", SEED)

## E. Decision rule and final untouched test

**Do not automatically select the largest profit improvement.** If on-time delivery is a required objective, favor a configuration with non-increasing modeled late probability (and adequate profit/days). If none exists, report the trade-off; do not claim the objective was met. Only after fixing weights and all other settings on validation should the test set be evaluated once.

In [ ]:
print('Scenarios meeting all three modeled directions on validation:')
display(summary.loc[summary['all_three_improve']].sort_values(['mean_delta_profit'],ascending=False))
print('No test results were used to select weights in this supplement.')
print('To evaluate a frozen scenario, train once with its selected weights and seed,')
print('then use candidate_table(test), add_reward(...), and paired_for(Q,...,test).')

### Reporting guidance

- The table of 25,886 / 27,078 is **validation**, because it was printed by the `val` evaluation in the original corrected notebook.
- Report bootstrap intervals for **paired Q-learning minus SARSA** outcomes, with signs interpreted appropriately.
- Report variability across independent training seeds separately; these are **not** independent observations of supply-chain performance.
- Compare policies against modeled historical actions on the **same paired eligible orders**. Do not call modeled gains realized profit or verified late-delivery improvements.
- Since the experiment uses terminal one-step decisions, Q-learning and SARSA update equations coincide; the comparison is a robustness check, not evidence of long-term sequential algorithm superiority.
- Important: the action space here is **shipping mode on the observed route**, not validated reassignment of fulfillment origin.
